# HR Employee Attrition Analysis
## Notebook 2 — Data Cleaning

**Author:** Fatimah Alradwan  
**Goal:** Clean the dataset and prepare it for analysis.

### What we will do in this notebook:
1. Drop useless columns that add no analytical value
2. Convert text columns to the right format
3. Create new useful columns for deeper analysis
4. Save the clean dataset to a new CSV file

---

In [1]:
# ── Import libraries ───────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries imported")

Libraries imported


In [2]:
# ── Load the raw dataset ───────────────────────────────────────────
df = pd.read_csv('../data/hr_raw.csv')

print(f"Raw data loaded: {df.shape[0]} rows × {df.shape[1]} columns")

Raw data loaded: 1470 rows × 35 columns


## Step 1 — Drop Useless Columns

Three columns have only one unique value across all 1,470 employees.
They carry zero information and will never help us find patterns.

- `EmployeeCount` — always 1
- `StandardHours` — always 80
- `Over18` — always Y

In [3]:
# ── Verify they are truly useless before dropping ──────────────────
print("EmployeeCount unique values :", df['EmployeeCount'].unique())
print("StandardHours unique values :", df['StandardHours'].unique())
print("Over18 unique values        :", df['Over18'].unique())

EmployeeCount unique values : [1]
StandardHours unique values : [80]
Over18 unique values        : ['Y']


In [4]:
# ── Drop the useless columns ───────────────────────────────────────
cols_to_drop = ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber']
df = df.drop(columns=cols_to_drop)

print(f"Dropped {len(cols_to_drop)} columns")
print(f"   Dataset is now: {df.shape[0]} rows × {df.shape[1]} columns")

Dropped 4 columns
   Dataset is now: 1470 rows × 31 columns


## Step 2 — Convert Text Columns to Category Type

Columns like Department, JobRole, and Gender are text (object type).
Converting them to 'category' type makes them:
- More memory efficient
- Easier to group and analyse
- Properly recognised by visualisation libraries

In [5]:
# ── Identify all text columns ──────────────────────────────────────
text_cols = df.select_dtypes(include='object').columns.tolist()
print("Text columns found:")
for col in text_cols:
    print(f"   {col} — {df[col].nunique()} unique values")

Text columns found:
   Attrition — 2 unique values
   BusinessTravel — 3 unique values
   Department — 3 unique values
   EducationField — 6 unique values
   Gender — 2 unique values
   JobRole — 9 unique values
   MaritalStatus — 3 unique values
   OverTime — 2 unique values


In [7]:
# ── Convert text columns to category type ─────────────────────────
for col in text_cols:
    df[col] = df[col].astype('category')

print("Converted to category type:")
print(df.select_dtypes(include='category').dtypes)

Converted to category type:
Attrition         category
BusinessTravel    category
Department        category
EducationField    category
Gender            category
JobRole           category
MaritalStatus     category
OverTime          category
dtype: object


## Step 3 — Create New Columns

We create three new columns that will make our analysis more meaningful:

1. `attrition_flag` — converts Yes/No text to 1/0 number so we can calculate rates
2. `age_group` — groups employees into age bands for comparison
3. `tenure_group` — groups employees by how long they've been at the company

In [8]:
# ── 1. attrition_flag: Yes → 1, No → 0 ────────────────────────────
df['attrition_flag'] = (df['Attrition'] == 'Yes').astype(int)

print("attrition_flag sample:")
print(df[['Attrition', 'attrition_flag']].head(10))

attrition_flag sample:
  Attrition  attrition_flag
0       Yes               1
1        No               0
2       Yes               1
3        No               0
4        No               0
5        No               0
6        No               0
7        No               0
8        No               0
9        No               0


In [14]:
# ── 2. age_group: bin employees into age bands ─────────────────────
df['age_group'] = pd.cut(
    df['Age'],
    bins=[17, 25, 35, 45, 60],
    labels=['18–25', '26–35', '36–45', '46–60']
)

print("Age group distribution:")
print(df['age_group'].value_counts().sort_index())

Age group distribution:
age_group
18–25    123
26–35    606
36–45    468
46–60    273
Name: count, dtype: int64


In [19]:
# ── 3. tenure_group: bin by years at company ──────────────────────
df['tenure_group'] = pd.cut(
    df['YearsAtCompany'],
    bins=[-1, 2, 5, 10, 40],
    labels=['0–2 yrs', '3–5 yrs', '6–10 yrs', '10+ yrs']
)

print("Tenure group distribution:")
print(df['tenure_group'].value_counts().sort_index())

Tenure group distribution:
tenure_group
0–2 yrs     342
3–5 yrs     434
6–10 yrs    448
10+ yrs     246
Name: count, dtype: int64


## Step 4 — Final Check and Save

We verify the clean dataset looks correct, then save it as a new file.
The clean file goes into data/hr_clean.csv.

In [20]:
# ── Final shape and column check ───────────────────────────────────
print(f"Final dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumns in clean dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:02d}. {col}")

Final dataset: 1470 rows × 34 columns

Columns in clean dataset:
   01. Age
   02. Attrition
   03. BusinessTravel
   04. DailyRate
   05. Department
   06. DistanceFromHome
   07. Education
   08. EducationField
   09. EnvironmentSatisfaction
   10. Gender
   11. HourlyRate
   12. JobInvolvement
   13. JobLevel
   14. JobRole
   15. JobSatisfaction
   16. MaritalStatus
   17. MonthlyIncome
   18. MonthlyRate
   19. NumCompaniesWorked
   20. OverTime
   21. PercentSalaryHike
   22. PerformanceRating
   23. RelationshipSatisfaction
   24. StockOptionLevel
   25. TotalWorkingYears
   26. TrainingTimesLastYear
   27. WorkLifeBalance
   28. YearsAtCompany
   29. YearsInCurrentRole
   30. YearsSinceLastPromotion
   31. YearsWithCurrManager
   32. attrition_flag
   33. age_group
   34. tenure_group


In [21]:
# ── Confirm no missing values crept in ────────────────────────────
missing = df.isnull().sum()
print("Missing values check:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values")

Missing values check:
No missing values


In [24]:
# ── Save clean dataset ─────────────────────────────────────────────
df.to_csv('../data/hr_clean.csv', index=False)
print("Clean dataset saved to data/hr_clean.csv")
print(f"   {df.shape[0]} rows × {df.shape[1]} columns ready for analysis")

Clean dataset saved to data/hr_clean.csv
   1470 rows × 34 columns ready for analysis
